# MIAPPE-API Python Demo

This notebook demonstrates the Python API for working with MIAPPE-compliant phenotyping metadata.

## 1. Getting Started

First, let's import the main components from miappe-api.

In [ ]:
from miappe_api.models import get_model
from miappe_api.validators import validate
from miappe_api.storage import JsonStorage, YamlStorage
from miappe_api.specs import SpecLoader

## 2. Exploring Available Entities

MIAPPE v1.1 defines 14 entities for describing phenotyping experiments. Let's list them.

In [ ]:
loader = SpecLoader()
entities = loader.list_entities(version="1.1")
print(f"MIAPPE v1.1 has {len(entities)} entities:\n")
for entity in entities:
    print(f"  - {entity}")

## 3. Working with Models

Models are dynamically generated from YAML specifications. Use `get_model()` to get a Pydantic model for any entity.

In [ ]:
# Get the Investigation model
Investigation = get_model("Investigation", version="1.1")

# Show the model's fields
print("Investigation fields:")
for name, field in Investigation.model_fields.items():
    required = "required" if field.is_required() else "optional"
    print(f"  {name}: {required}")

### Creating an Investigation

In [ ]:
import datetime

# Create an investigation instance
investigation = Investigation(
    unique_id="INV-2024-001",
    title="Drought Tolerance in Wheat Varieties",
    description="A multi-year study examining drought tolerance traits across 50 wheat varieties under controlled water stress conditions.",
    submission_date=datetime.date(2024, 3, 15),
    license="https://creativecommons.org/licenses/by/4.0/",
    miappe_version="1.1",
)

print("Investigation created:")
print(f"  ID: {investigation.unique_id}")
print(f"  Title: {investigation.title}")
print(f"  Submission: {investigation.submission_date}")

### Serialization

Models can be serialized to dictionaries and JSON.

In [ ]:
# Convert to dictionary
data = investigation.model_dump(exclude_none=True)
print("As dictionary:")
data

In [ ]:
# Convert to JSON
json_str = investigation.model_dump_json(indent=2, exclude_none=True)
print("As JSON:")
print(json_str)

### JSON Schema

Get the JSON Schema for validation or documentation.

In [ ]:
schema = Investigation.model_json_schema()
print("Required fields:", schema.get("required", []))
print("\nProperties:")
for prop, details in list(schema["properties"].items())[:5]:
    print(f"  {prop}: {details.get('type', 'any')}")

## 4. Working with Studies

Studies are the core experimental unit within an investigation.

In [ ]:
Study = get_model("Study", version="1.1")

study = Study(
    unique_id="STU-2024-001",
    title="Field Trial 2024 - Site A",
    description="First year field trial at research station A",
    start_date=datetime.date(2024, 4, 1),
    end_date=datetime.date(2024, 9, 30),
    geographic_location="Germany",
    latitude=52.5200,
    longitude=13.4050,
    experimental_design_type="Randomized Complete Block Design",
    growth_facility_type="field",
)

print(f"Study: {study.title}")
print(f"Duration: {study.start_date} to {study.end_date}")
print(f"Location: {study.geographic_location} ({study.latitude}, {study.longitude})")

## 5. Biological Materials

Biological materials represent the germplasm or plant material used in studies.

In [ ]:
BiologicalMaterial = get_model("BiologicalMaterial", version="1.1")

materials = [
    BiologicalMaterial(
        unique_id="BM-001",
        organism="Triticum aestivum",
        genus="Triticum",
        species="aestivum",
        infraspecific_name="cv. Apache",
        accession_number="PI 12345",
    ),
    BiologicalMaterial(
        unique_id="BM-002",
        organism="Triticum aestivum",
        genus="Triticum",
        species="aestivum",
        infraspecific_name="cv. Soissons",
        accession_number="PI 12346",
    ),
]

print("Biological Materials:")
for mat in materials:
    print(f"  {mat.unique_id}: {mat.organism} {mat.infraspecific_name}")

## 6. Observed Variables

Observed variables follow the Crop Ontology model: Trait + Method + Scale.

In [ ]:
ObservedVariable = get_model("ObservedVariable", version="1.1")

variables = [
    ObservedVariable(
        unique_id="VAR-001",
        name="Plant Height",
        trait="Plant height",
        trait_accession_number="TO:0000207",
        method="Ruler measurement from soil surface to highest point",
        scale="cm",
        time_scale="Days after sowing",
    ),
    ObservedVariable(
        unique_id="VAR-002",
        name="Leaf Wilting Score",
        trait="Leaf wilting",
        trait_accession_number="TO:0000109",
        method="Visual scoring on 1-9 scale",
        scale="1-9 ordinal scale",
    ),
]

print("Observed Variables:")
for var in variables:
    print(f"  {var.name}")
    print(f"    Trait: {var.trait} ({var.trait_accession_number})")
    print(f"    Method: {var.method}")
    print(f"    Scale: {var.scale}")
    print()

## 7. Validation

The `validate()` function checks data against MIAPPE rules beyond basic type checking.

In [ ]:
# Valid data
valid_data = {
    "unique_id": "INV-001",
    "title": "My Investigation",
}

errors = validate(valid_data, "investigation", version="1.1")
if errors:
    print("Validation errors:")
    for e in errors:
        print(f"  {e}")
else:
    print("Validation passed!")

In [ ]:
# Invalid data - missing required field
invalid_data = {
    "unique_id": "INV-001",
    # missing title
}

errors = validate(invalid_data, "investigation", version="1.1")
print("Validation errors:")
for e in errors:
    print(f"  {e.field}: {e.message}")

In [ ]:
# Invalid data - bad ID format
bad_id_data = {
    "unique_id": "INV 001 with spaces",  # spaces not allowed
    "title": "My Investigation",
}

errors = validate(bad_id_data, "investigation", version="1.1")
print("Validation errors:")
for e in errors:
    print(f"  {e.field}: {e.message}")

In [ ]:
# Invalid data - date range error
bad_dates = {
    "unique_id": "STU-001",
    "title": "Study with bad dates",
    "start_date": datetime.date(2024, 12, 31),
    "end_date": datetime.date(2024, 1, 1),  # end before start!
}

errors = validate(bad_dates, "study", version="1.1")
print("Validation errors:")
for e in errors:
    print(f"  {e.field}: {e.message}")

## 8. Storage

Save and load entities to JSON or YAML files.

In [ ]:
from pathlib import Path
import tempfile

# Create a temporary directory for our files
temp_dir = Path(tempfile.mkdtemp())
print(f"Working directory: {temp_dir}")

### JSON Storage

In [ ]:
json_storage = JsonStorage(indent=2)

# Save investigation to JSON
json_path = temp_dir / "investigation.json"
json_storage.save(investigation, json_path)
print(f"Saved to: {json_path}")
print("\nFile contents:")
print(json_path.read_text())

In [ ]:
# Load investigation from JSON
loaded_inv = json_storage.load(json_path, Investigation)
print(f"Loaded: {loaded_inv.title}")
print(f"ID matches: {loaded_inv.unique_id == investigation.unique_id}")

### YAML Storage

In [ ]:
yaml_storage = YamlStorage()

# Save study to YAML
yaml_path = temp_dir / "study.yaml"
yaml_storage.save(study, yaml_path)
print(f"Saved to: {yaml_path}")
print("\nFile contents:")
print(yaml_path.read_text())

In [ ]:
# Load study from YAML
loaded_study = yaml_storage.load(yaml_path, Study)
print(f"Loaded: {loaded_study.title}")
print(f"Location: {loaded_study.geographic_location}")

## 9. Exploring Specifications

You can explore the underlying YAML specifications directly.

In [ ]:
# Load the Investigation specification
spec = loader.load_entity("investigation", version="1.1")

print(f"Entity: {spec.name}")
print(f"Version: {spec.version}")
print(f"Ontology term: {spec.ontology_term}")
print(f"\nDescription:\n{spec.description}")

In [ ]:
# List required vs optional fields
required = spec.get_required_fields()
optional = spec.get_optional_fields()

print(f"Required fields ({len(required)}):")
for f in required:
    print(f"  - {f.name}: {f.type.value}")

print(f"\nOptional fields ({len(optional)}):")
for f in optional[:5]:  # Show first 5
    print(f"  - {f.name}: {f.type.value}")
if len(optional) > 5:
    print(f"  ... and {len(optional) - 5} more")

In [ ]:
# Examine field details including ontology terms
print("Field details with ontology references:\n")
for field in spec.fields[:5]:
    print(f"{field.name}:")
    print(f"  Type: {field.type.value}")
    print(f"  Required: {field.required}")
    if field.ontology_term:
        print(f"  Ontology: {field.ontology_term}")
    if field.constraints:
        print(f"  Constraints: {field.constraints}")
    print()

## 10. Complete Example: Building an Investigation

Let's put it all together and build a complete investigation structure.

In [ ]:
# Get all the models we need
Investigation = get_model("Investigation", version="1.1")
Study = get_model("Study", version="1.1")
Person = get_model("Person", version="1.1")
BiologicalMaterial = get_model("BiologicalMaterial", version="1.1")
ObservedVariable = get_model("ObservedVariable", version="1.1")
Factor = get_model("Factor", version="1.1")
FactorValue = get_model("FactorValue", version="1.1")

In [ ]:
# Create researchers
researchers = [
    Person(
        name="Dr. Jane Smith",
        email="j.smith@university.edu",
        institution="University Research Center",
        role="Principal Investigator",
        orcid="0000-0001-2345-6789",
    ),
    Person(
        name="Dr. John Doe",
        email="j.doe@university.edu",
        institution="University Research Center",
        role="Data Manager",
    ),
]

# Create experimental factors
factors = [
    Factor(
        unique_id="FACTOR-001",
        name="Water Treatment",
        description="Water stress treatment applied during vegetative growth",
        factor_type="treatment",
    ),
]

# Create factor values
factor_values = [
    FactorValue(
        unique_id="FV-001",
        factor_id="FACTOR-001",
        value="Control (100% ET)",
        description="Well-watered control at 100% evapotranspiration",
    ),
    FactorValue(
        unique_id="FV-002",
        factor_id="FACTOR-001",
        value="Moderate stress (50% ET)",
        description="Moderate water stress at 50% evapotranspiration",
    ),
    FactorValue(
        unique_id="FV-003",
        factor_id="FACTOR-001",
        value="Severe stress (25% ET)",
        description="Severe water stress at 25% evapotranspiration",
    ),
]

In [ ]:
# Create the complete investigation
complete_investigation = Investigation(
    unique_id="INV-DROUGHT-2024",
    title="Multi-Environment Drought Tolerance Assessment in European Wheat",
    description="A comprehensive study evaluating drought tolerance mechanisms "
                "across 50 European wheat cultivars under controlled water stress "
                "conditions at three experimental sites.",
    submission_date=datetime.date(2024, 6, 1),
    public_release_date=datetime.date(2025, 1, 1),
    license="https://creativecommons.org/licenses/by/4.0/",
    miappe_version="1.1",
    associated_publications=[
        "doi:10.1234/example.2024.001",
    ],
)

print("Complete Investigation:")
print(f"  Title: {complete_investigation.title}")
print(f"  ID: {complete_investigation.unique_id}")
print(f"  Submission: {complete_investigation.submission_date}")
print(f"  Release: {complete_investigation.public_release_date}")

In [ ]:
# Save the complete investigation
output_path = temp_dir / "complete_investigation.yaml"
yaml_storage.save(complete_investigation, output_path)

print(f"Investigation saved to: {output_path}")
print("\nYAML content:")
print(output_path.read_text())

## Summary

This notebook demonstrated:

1. **Exploring entities** - List available MIAPPE entities and their fields
2. **Creating models** - Use `get_model()` to get Pydantic models dynamically
3. **Serialization** - Convert models to dict, JSON, and get JSON schemas
4. **Validation** - Validate data with `validate()` for business rules
5. **Storage** - Save/load with `JsonStorage` and `YamlStorage`
6. **Specifications** - Access underlying YAML specs with `SpecLoader`

For more information, see the [documentation](https://miappe-api.readthedocs.io).